# Knee MRI — preprocessing & model pipeline (PSEUDOCODE PLAN)

**Status:** design only. Nothing here executes. Every code cell is a skeleton — real function
names and signatures, comment-only bodies — so the four of us can split the work and code
against fixed contracts without blocking each other.

**Competition:** [rsna-knee-abnormality-detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)
· deadline 2026-10-22 · metric = mean of 12 per-column ROC AUCs.

---

## 0. Reality check — five things we had wrong

Correcting these now is cheaper than correcting them in week 3.

| # | What we assumed | What is actually true |
|---|---|---|
| 1 | "Each **series** gives 3 arrays: X / Y / Z, ~40 images each" | A series is **one** acquisition in **one** plane — that is why `train_series.csv` has an `Anatomical_Plane` column. A study averages **~5.5 series**. Our unit of work is the **study**, reduced to **4 fixed slots** (§2). Reslicing one plane into the others fabricates resolution: through-plane is 3–4 mm vs 0.3–0.6 mm in-plane, and the study already contains a real acquisition in each plane. |
| 2 | "One specialised **model** per label" | Right instinct, wrong unit. 12 fine-tuned backbones × ~1,300 test studies will not fit the **9-hour** GPU limit. One shared encoder + **12 specialised heads** gives each label its own attention, threshold and colour map at 1/12 of the cost. See §5. |
| 3 | "Train on images only, ignore reports, don't trust the 58" | With the 58 excluded and reports excluded there are **zero** labels. Resolution: the reports become a **label file**, not a module — see §0.1. |
| 4 | "The model returns a knee with a colour spot on the injury" | There are **no masks and no boxes** in this dataset, only study-level 0/1. The colour must come from **weakly-supervised** localisation (CAM / MIL attention). Cheaper to build, but coarse and not always right — §7 gates it. |
| 5 | "Download the data" | It is **569 GB**, and our competition-download endpoint is 429-locked. We use a **17.4 GB public preprocessed mirror** instead, and Kaggle GPU for the heavy pass. §2. |

### 0.0 ⛔ Action item zero — `data/` is NOT gitignored on this branch

Verified just now:

```console
$ grep -n data .gitignore
4:.data/                          # ← leading dot. Does not match data/
$ git check-ignore -q data/train.csv ; echo $?
1                                 # ← 1 = NOT IGNORED
```

The moment anyone runs the download and types `git add .`, gigabytes of DICOM go into git
history. **`niko/fix/notebookKaggle00` already contains the fix** (`data/`, `!data/.gitkeep`,
`notebooks/data/`).

> **Merge that branch before any code writes a byte into `data/`.**

### 0.1 Where supervision actually comes from

- **58 of 4,407** studies have all 12 labels. The other **4,349** have a `Report` and nothing else.
- The 58 are **radiologist-adjudicated** — two MSK radiologists plus an adjudicator, the same
  process that produced the hidden test ground truth. They are the *most* reliable labels we have.
- But every one of the 58 has ≥1 positive finding, so the set is **selection-biased**, and 58
  across folds is ~12 per fold where the standard error of an AUC is ≈0.15.
  → **Never train on them. Always audit against them.**
- `test.csv` has **no Report column**. Any text branch is a *training-time teacher only*.
  Treating report text as a test feature is the classic fatal error in this competition.
  Module `M3` enforces this with an assertion (§4).

---
## 1. Architecture at a glance

```
  ┌─────────────── KAGGLE ONLY — 569 GB already mounted at /kaggle/input ───────────────┐
  │                                                                                     │
  │  train_series.csv ─► M1 assign_slots ─► slots.csv ─► M2 dicom→volume ─► volumes.npy │
  │   24,371 rows          pandas, 30 s      4407 × 4      CPU kernel        uint8       │
  │   plane + fluid        NO dicom reads                  resumable      (4,24,224,224) │
  │                                                                            │        │
  │                                                    M4 encode  ◄────────────┘        │
  │                                                 GPU, DINOv2-S frozen                │
  └────────────────────────────────────────────────────────┬────────────────────────────┘
                                                           ▼
   train.csv ─► M3 label factory ─► soft_targets.csv   features.npy  [N,4,K,D]  ~600 MB
    (Report)    merge public LLM     Y (soft) + W       published as a Kaggle dataset,
                label CSVs           (W=0 ⇒ masked)     then pulled to the laptops
                                          │                    │
                                          └─────────┬──────────┘
                                                    ▼
                              M5 heads   —   LAPTOP, ~20 s per experiment
                              StandardScaler → PCA(512) → 12 × weighted Ridge
                                                    │
                           ┌────────────────────────┴────────────────────────┐
                           ▼                                                 ▼
              submit kernel (GPU, internet OFF)                  M5 cam ─► M6 render packs
              decodes the HIDDEN test from DICOM                 exact linear decomposition
                           │                                                 │
                           ▼                                                 ▼
                    submission.csv                              app/  2D tri-planar + 3D
                   (Kaggle: 12 AUCs)                            + 12 confidence bars
```

**The one principle that makes this fast: encode once, iterate forever.**
The DINOv2 pass over ~203k slices is the only expensive step. It runs **~3 times for the whole
project**. Everything downstream reads a cached `features.npy`, so a head trains in **seconds**.
That is what makes 12 specialists affordable — and what lets us try 50 ideas instead of 5.

> ⚠️ **The cache is a *training* device only.** The scoring kernel gets a hidden test set it has
> never seen and must run **DICOM → preprocess → encode → submission from scratch, inside 9 hours,
> with no internet**. That path shares code with training but caches nothing.
> **Write the scoring kernel skeleton before the training loop** (Stage 0, §6) — a pipeline that
> trains beautifully and cannot submit is worth zero.

> 💡 **Do not download 569 GB — develop on Kaggle.** In a Kaggle notebook the full competition is
> already mounted at `/kaggle/input/`. M2 should be authored *there* from hour one, against real
> DICOM, and only then ported to `src/`. Local tiers exist for offline work, not as a prerequisite.

### Repo layout to create

```
src/stk/
  paths.py      find_repo_root(), comp_root(), where() -> 'kaggle'|'colab'|'local'
  data.py    M1 frozen constants, preflight(), census(), assign_slots()
  preprocess.py M2 series_to_volume() — the ONE function that runs identically in the
                   training build AND in the scoring kernel. Never fork it.
  labels.py  M3 merge public LLM label CSVs -> Y, W, folds
  models.py  M4 encode (GPU) + pool + Ridge heads (laptop) + submission
  cam.py     M5 exact linear-decomposition attention (no Grad-CAM, no backward pass)
  viz.py     M6 render packs, tri-planar viewer, MIP GIF
app/          streamlit_app.py
notebooks/    thin drivers that import src/stk — no logic lives in notebooks
```

---
## 2. Data contracts

Every teammate codes against this table. If a contract changes, it changes **here first**.

| Artifact | Path | Shape / dtype | Size |
|---|---|---|---|
| competition CSVs | `data/*.csv` | verbatim | ~25 MB |
| LLM report labels | `data/labels_llm_*.csv` | 4,407 × 13 | <1 MB |
| series manifest | `data/manifest/manifest_series.parquet` | 24,371 rows | ~8 MB |
| **canonical volume** | `data/processed/<profile>/<study>/<series>.npy` | `uint8 (24,224,224)` | 1.2 MB |
| slot assignment | `data/manifest/slots.csv` | 4,407 × 5 | 600 KB |
| study tensor | *(in memory, built from 4 slots)* | `uint8 (4,24,224,224)` | — |
| **soft targets** | `data/derived/soft_targets.csv` | 4,407 × 40 | ~2 MB |
| label audit | `data/derived/label_audit.csv` | 12 × 8 | 2 KB |
| folds | `data/derived/study_folds.csv` | 4,407 × 4 | 400 KB |
| **cached features** | `data/features/features_<tag>.npy` | `float16 [N,4,K,D]` | ~600 MB |
| feature mask | `data/features/mask_<tag>.npy` | `bool [N,4,K]` | 265 KB |
| heat maps | `data/_heat/<study>.npz` | `uint8 (12,S,gh,gw)` | ~14 KB |
| **render pack** | `data/render_packs/<study>.npz` | `gray_iso uint8 128³`<br>`heat_iso uint8 (12,64³)` | ~5 MB |
| submission | `/kaggle/working/submission.csv` | 1,300 × 13 | ~150 KB |

**Frozen constants** — live in `src/stk/data.py`, imported everywhere. Changing one invalidates
every cache, which is why `CACHE_TAG` encodes them.

```python
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
           "Medial OA", "Lateral OA", "PF OA", "Effusion",
           "Synovitis", "Baker's", "Contusion", "Fracture"]   # submission order. NEVER sort.

# A study averages ~5.5 series. We do not take "3 planes" — we take 4 fixed SLOTS,
# because fluid-sensitive vs structural matters as much as the plane:
#   Contusion / Fracture / Effusion are invisible outside a fluid-sensitive sequence.
SLOTS = [("SAG_FLUID",  "Sagittal", 1),   # 94.2% of studies have one
         ("SAG_STRUCT", "Sagittal", 0),   # 96.8%
         ("COR_FLUID",  "Coronal",  1),   # 96.4%
         ("AX_FLUID",   "Axial",    1)]   # ~100%
# AX_STRUCT is DROPPED — present for only 19.4% of studies, so the head would spend most
# of its capacity learning "this slot is missing".
# Fat_Suppression is DROPPED — byte-identical to Fluid_Sensitive across all 24,371 rows,
# despite the official docs claiming otherwise. Re-assert on the test manifest, warn not fail.

DEPTH, SIZE, CROP_MM = 24, 224, 130.0   # 0.580 mm/px. 224 = 16x14 -> DINOv2 ViT-S/14 compatible
WINDOW_PCT = (0.5, 99.5)                # per SERIES, on the cropped foreground
CACHE_TAG  = "v1_d24_s224_c130_4slot"   # in every filename; a mismatched tag REFUSES to load
```

**Serialisation rule:** `.npy` / `.npz` / `.csv` only. **Never pickle.** A pickle written by
one teammate's sklearn version is a silent failure on everyone else's.

---
## 3. Module pseudocode

Six modules in dependency order. `# STRETCH` marks anything droppable without breaking the chain.

### M1 — Data access & materialisation
**goal** get from "569 GB on Kaggle, 429-locked" to a working local dataset
**in** Kaggle credentials · **out** `data/*.csv`, `manifest_series.parquet`, tiered volumes

In [ ]:
# src/data/kaggle_pull.py

def preflight():
    """FIRST CELL OF EVERY NOTEBOOK. Raises, never warns.
    1. subprocess git check-ignore data/x -> if not ignored, RAISE with the exact fix:
         "git merge niko/fix/notebookKaggle00"     (see section 0.0)
    2. shutil.disk_usage(repo).free >= 20 GiB
    3. print numpy/pandas/sklearn/torch versions + CACHE_TAG
    4. print where() -> 'kaggle' | 'colab' | 'local'
    """

def census(train_df, series_df):
    """30 SECONDS OF PANDAS. Run this BEFORE writing any other code — it retires four
    assumptions that this whole plan rests on, and needs no DICOM and no download:
      a. series_df.groupby(['Anatomical_Plane','Fluid_Sensitive']).StudyInstanceUID.nunique()
         / 4407  -> the REAL per-slot fill rate. Drop any slot below 0.90.
      b. (series_df.Fluid_Sensitive == series_df.Fat_Suppression).all()   -> is it really dupe?
      c. train_df[TARGETS].notna().all(axis=1).sum()   -> print it, do NOT assert == 58
      d. train_df.Report.duplicated().sum()            -> identical reports leak across folds
    """

TIERS = {
    "T0_meta":  "the 5 competition CSVs + LLM label CSVs        ~25 MB   laptop",
    "T1_dicom": "8 studies of RAW DICOM — the only way to test  ~1 GB    laptop",
    "T2_gold":  "the 58 adjudicated studies (audit set)         ~1.4 GB  laptop",
    "T3_full":  "public 17.4 GB preprocessed .npz mirror        17 GB    Kaggle/Colab",
}

def authenticate():
    """Reuse the .env convention from kaggleFetch.ipynb: KAGGLE_USERNAME + KAGGLE_API_TOKEN."""

def pull_tier(tier, dest="data/"):
    """Resumable, rate-limit-safe.
    - skip any file already on disk with the right sha256  -> reruns are free
    - exponential backoff on 429; NEVER parallelise (parallel just burns the quota)
    - IMPORTANT: competition_download_file() is 429-locked for us, but
      dataset_download_files() is throttled SEPARATELY and works. Prefer datasets.
    """

def build_manifest(split="train"):
    """One row per series. This is the index everything downstream reads —
    nothing else is ever allowed to walk the filesystem.
    cols: study_uid, series_uid, plane, fluid_sensitive, n_slices, rows, cols,
          pixel_spacing, slice_thickness, laterality, transfer_syntax
    NOTE: Fat_Suppression is byte-identical to Fluid_Sensitive across all 24,371
          rows despite the docs claiming otherwise -> keep one, drop the other.
    """

def verify_tier(tier):
    """sha256 every file against DATASETS.lock.json so all four of us have identical bytes."""

### M2 — DICOM → canonical volume  *(the core preprocessing)*
**goal** every study becomes one comparable `uint8 (3,24,224,224)` tensor
**in** a series directory · **out** `data/processed/<profile>/<study>/<series>.npy`

Five traps live in this module. Each line below exists because of one.

In [ ]:
# src/preprocess/dicom_to_volume.py

def read_series(series_dir):
    """Filenames are bare DICOM UIDs — they carry NO ordering information.
    TRAP 1 (ordering): sort by ImagePositionPatient projected on the slice normal.
                       NOT by filename (uncorrelated with anatomy), NOT by InstanceNumber alone.
    TRAP 2 (codecs):   mixed transfer syntaxes incl. JPEG-Lossless and JPEG-2000.
                       bare pydicom raises on .pixel_array -> pylibjpeg plugins are MANDATORY.
    Apply RescaleSlope/Intercept; invert MONOCHROME1; handle signed pixel data.
    returns RawSeries(vol int16 (S,H,W), spacing (dz,dy,dx) mm, dirs (3,3), meta)
    """

def normalize_intensity(vol):
    """TRAP 3: MRI has NO Hounsfield units — CT windowing is meaningless here.
    ONE percentile window (1st–99.5th) computed per SERIES from POOLED slice pixels.
    Per-SLICE windowing destroys the relative-brightness signal that Effusion /
    Synovitis / Baker's are literally made of.
    -> uint8 0-255
    """

def crop_to_physical_extent(vol, spacing, mm=130):
    """TRAP 4: PixelSpacing varies 3.4x across the corpus (FOV 70-320 mm, median 160).
    Resizing to a fixed PIXEL count does not normalise physical scale — a 512px crop
    means different millimetres in different studies. Crop to a constant MM extent first,
    then resize. 130 mm sits below the FOV of 99.6% of series.
    ! Baker's cysts live in the popliteal fossa and a tight crop cuts them off.
      -> that is why the 'context' profile (full FOV) exists. See §5.
    """

def resolve_study_laterality(study_id, series_df):
    """Resolve the side ONCE PER STUDY, before any series is preprocessed.
    If each series decides for itself, one study can end up with a flipped sagittal and an
    un-flipped coronal — the planes then disagree about which side is medial, which is worse
    than not flipping at all.
    Vote across every series in the study; fall back to ImageOrientationPatient sign.
    returns (side in {'L','R',None}, source, confident: bool)
    """

def fix_laterality(vol, plane, side, confident):
    """TRAP 5, and it silently costs a third of the metric.
    4 of 12 labels are side-specific (Medial/Lateral Meniscus, Medial/Lateral OA) and the
    corpus mixes left and right knees. The DICOM Laterality tag is MISSING on ~47% of studies.
      coronal / axial -> flip the last pixel axis
      sagittal        -> reverse the STACK order, do not touch pixels
      not confident   -> LEAVE IT ALONE. A wrong flip is worse than no flip.
    Record `side` in the manifest either way, and give the head a laterality channel so it
    can learn the unflipped cases rather than being lied to.
    """

def preprocess_series(series_dir, profile, side, confident, csv_plane=None):
    """read -> normalize -> crop -> resize -> fix_laterality
    Downsampling uses cv2.resize(..., INTER_AREA), NOT scipy.ndimage.zoom — ~10x faster and
    correctly area-averages, which matters when the 9-hour scoring kernel does this 1,300 times.
    Pure: no writes, no network. Safe inside multiprocessing.Pool, unit-testable.
    returns (uint8 (24,224,224) | None, meta)   # None = failure, still gets a manifest row
    meta MUST carry per-series `mm_per_px` and `slice_spacing_mm` — they vary 3.4x across the
    corpus, so they can never be module constants. M6 needs them to draw a scale bar honestly.
    """

def assign_slots(series_df):
    """Decide which 4 series of each study we will EVER look at — from train_series.csv alone,
    no DICOM reads, ~30 s of pandas for the whole corpus. Ties broken by slice count.
    -> slots.csv: study_uid, SAG_FLUID, SAG_STRUCT, COR_FLUID, AX_FLUID (series_uid or NaN)
    """

def build_study_tensor(study_id, slots_row, profile):
    """Stack the 4 slots in SLOTS order.
    returns (uint8 (4,24,224,224), present_mask uint8 (4,), meta)
    Missing slot -> zeros + present_mask[i]=0. The head must learn to IGNORE it, never impute:
    an imputed slot teaches the model that a missing sequence looks like an average knee.
    """

PROFILES = {
    "default": dict(mm=130, size=224, slices=24),   # full corpus, ~16 GB
    "hires":   dict(mm=130, size=384, slices=32),   # gold-58 only, local dev
    "context": dict(mm=200, size=224, slices=24),   # STRETCH: full FOV for Baker's / posterior effusion
}

### M3 — Supervision: reports → soft targets  *(the label factory)*
**goal** turn 4,349 report-only studies into trainable soft labels **with a trust weight**
**in** `train.csv`, public LLM label CSVs · **out** `soft_targets.csv`, `label_audit.csv`

This is the highest-leverage module in the project and the one most likely to be built wrong.

In [ ]:
# src/labels/report_labels.py

def load_public_llm_labels():
    """We do NOT build an NLP pipeline. Several teams published LLM-derived labels for all
    4,407 studies as Kaggle DATASETS (<1 MB, and the dataset endpoint is not 429-locked):
        pilkwang/rsna-knee-llm-labels        -> report_labels_v2.csv  (has __conf + __verdict!)
        stevenleehans/rsna-knee-report-labels
    report_labels_v2 already ships per-label confidence and a YES/NO/UNK verdict, which is
    exactly the trust signal we need. Blend 2-3 sources by rank-mean.
    """

def audit_against_gold(soft, gold_58):
    """THE experiment that answers 'can we trust the labels?' with numbers instead of opinion.
    The 58 gold studies are the ONLY rows where a report AND an adjudicated label both exist.
    per label:
        agreement = (report_label == gold_label).mean()
        false_pos = report says yes, radiologist says no  -> negation-parsing failure
        false_neg = report silent, radiologist says yes    -> finding never dictated
    Overall agreement is ~82%, and it is VERY uneven per finding.
    Caveat: n=58 with imbalance -> treat as ranking guidance, not precise estimates.
    -> label_audit.csv
    """

def trust_weights(audit):
    """Turn the audit into a per-(study, label) weight matrix W.
    Measured silence rates (report never mentions the finding):
        Medial Meniscus  5.7%   ACL 8.1%   MCL 9.8%   Effusion 9.9%    <- train hard
        PF OA 18.5%   Contusion 20.8%   Medial OA 25.5%   Lateral OA 33.1%
        Baker's 45.9%   Fracture 56.4%   Synovitis 84.2%                <- barely supervised

    CRITICAL: a SILENT cell gets W = 0, not a soft prior of 0.28.
    "The report has no opinion" is a MISSING label, and the mathematically correct handling
    is masked / partial-label BCE — drop the term from the loss. Feeding it a prior teaches
    the model the base rate instead of the anatomy, which is exactly the wrong lesson for a
    label like Synovitis where 84% of cells are silent.
    -> W float32 (N,12), Y float32 (N,12)
    """

def make_folds(train, n=5):
    """StratifiedGroupKFold grouped by StudyInstanceUID.
    De-duplicate on report_md5 first — identical reports across studies leak between folds.
    The 58 gold studies are NEVER in a training fold. They are the audit set, full stop.
    -> study_folds.csv, imported by M4 and M5 so everyone uses the SAME split.
    """

def assert_label_factory_only(module):
    """THE GUARD. test.csv has no Report column. If report-derived anything reaches the
    inference path, our CV looks brilliant and the leaderboard collapses.
    assert no artifact from this module is importable by src/models/infer.py
    """

### M4 — Encode once: frozen backbone → cached features
**goal** the one expensive pass, run ~3× for the whole project
**in** study tensors · **out** `features_<tag>.npy float16 [N,4,K,D]`

In [ ]:
# src/models/encode.py

BACKBONES = {   # all must be pre-attached Kaggle Models — scoring kernels have NO internet
    "dinov2_s":   "metaresearch/dinov2/PyTorch/small/1     384-d, strongest frozen features",
    "effnet_b3":  "timm/tf-efficientnet/PyTorch/b3/1       cheap, proven 2.5D baseline",
    "radimagenet":"marwanmath/resnet-50-radimagenet        medical-domain pretrain, best transfer",
}

def to_model_input(vol_uint8):
    """grayscale -> 3ch. TRAP: build the 3 channels from ADJACENT SLICES (2.5D), not by
    repeating one slice — free 3D context at zero cost. Use the backbone's own normalisation.
    """

def encode_all(studies, backbone="dinov2_s", tag="v1"):
    """for each study, each SLOT, each of K slices: forward, keep [cls | mean_patch | max_patch].
    ENCODE THE TEST SET FIRST and never subsample it — a half-encoded test set is a dead run.
    Write float16 (halves the file, costs nothing measurable in AUC).
    -> features_<tag>.npy [N,4,K,D], mask_<tag>.npy [N,4,K]
    """

def cache_is_stale(tag):
    """Hash (profile, backbone, K, size) into the tag. A silent stale cache is the single
    most expensive bug available to us — it looks like a bad idea rather than a bad file."""

### M5 — The 12 specialist heads
**goal** each label gets its own model, sharing one encoder
**in** cached features + soft targets · **out** OOF predictions, `submission.csv`

This is §5 in code form — read that section first for *why* each head differs.

In [ ]:
# src/models/heads.py

def pool_features(feats, mask):
    """[N,4,K,D] -> [N, 4*2*D]  by mean+max over the K slices of each slot, masked.
    Foreground-masked top-k pooling is the upgrade; do the simple version first.
    """

def fit_heads(X, Y, W, folds):
    """START WITH RIDGE, NOT A NEURAL HEAD.
        StandardScaler -> PCA(512) -> 12x sample-weighted RidgeCV
    A full 5-fold, 12-label experiment costs ~20 SECONDS on a laptop with no GPU.
    That is the whole point of the cache: 50 experiments in a week, not 5.
    Only move to an attention head (below) once Ridge has plateaued and the LB confirms it.
    -> head_scaler.npz, head_pca.npy, head_ridge.npz   (.npy/.npz only — never a pickle)
    """

class SpecialistHead(nn.Module):   # STRETCH — only after Ridge plateaus
    """ONE per label. Still tiny — trains in seconds on cached features.
      attention over slices -> which slices matter for THIS finding
      attention over slots  -> ACL reads SAG_FLUID, PF OA reads AX_FLUID, MCL reads COR_FLUID
      linear -> 1 logit
    The slot/slice attention IS the localisation signal M6 consumes. Free, no extra model.
    """

def train_head(label, feats, mask, y, w, folds):
    """loss = (BCE(pred, y_soft) * W[:, label]).sum() / W[:, label].sum()   # masked BCE
    W == 0 rows contribute NOTHING — that is the point (see M3.trust_weights).
    Per-label knobs — this is where 'a specialised model per label' actually lives:
      - plane prior      (initialise plane attention toward the plane that owns the anatomy)
      - slice window     (restrict attention to a plausible slice range)
      - profile          ('context' full-FOV for Baker's; 'default' for the rest)
      - pos_weight       (Fracture is 7.2% prevalent — without this it predicts all-negative)
    """

def validate(oof, gold_58):
    """READ THIS BEFORE TRUSTING ANY NUMBER.
    OOF AUC on the 4,349 pseudo-labelled studies measures agreement with the REPORTS, not with
    the radiologists. It is a REGRESSION GUARD ("did I break something?"), never a score
    estimate — report labels agree with adjudicated truth only ~82%.
    The gold 58 are directional only (n~12/fold, AUC SE ~0.15).
    The leaderboard is the only real validation. We get 5 submissions/day — spend them.
    """

def fuse_and_submit(oof, test_preds):
    """The metric reads ONLY rank order:
      - calibration is worthless; do not waste time on it
      - ensemble by RANK mean, never probability mean
      - every label is worth exactly 1/12 -> a label left at chance forfeits ~0.029.
        Synovitis and Fracture deserve MORE attention than Effusion, not less.
    -> /kaggle/working/submission.csv, columns EXACTLY ['StudyInstanceUID'] + TARGETS
    """

### M6a — Weakly-supervised localisation (the colour)
**goal** a 3D heat volume per label, honestly gated
**in** features + head weights · **out** `data/_heat/<study>.npz`, `display_config.json`

In [ ]:
# src/viz/localize.py

def contribution_map(patch_tokens, coef_label):
    """Because the head is LINEAR, the attention map is exact, not an approximation:
    the logit is literally sum over patches of <token_p, coef>, so the per-patch
    contribution IS the map. One numpy einsum.
    NO Grad-CAM, NO backward pass, NO second model, NO GPU.
    -> signed (S, gh, gw); keep the sign — negative evidence is real information

    Honest resolution: DINOv2/14 at 224px on a 130 mm crop = ~8 mm per token. That is
    coarser than the ACL is thick. Label it 'model attention', never 'the lesion'.
    """

def heat_volume(cam, alpha_slices, vol_shape):
    """weight each slice's CAM by that head's slice attention, upsample to (S,224,224),
    smooth, mask to foreground (a CAM lighting up the air outside the knee is a bug you
    will only notice if you look).
    """

def qc_on_gold_58(heats, gold):
    """The honesty gate. For each label:
      anatomy_lift        does the ACL heat land in the intercondylar notch?
      plane_agreement_mm  do sagittal and coronal point at the SAME place in patient space?
      randomised_head     a shuffled head must score ~0 lift. If it doesn't, the CAM is noise.
    -> overlay_enabled per label. A label that fails QC ships WITHOUT a colour overlay.
    """

DISPLAY_CONTRACT = """
  overlay alpha is ALWAYS multiplied by the predicted confidence
  confidence below tau -> NO overlay at all, not a faint one
  the app returns 12 percentages + colour. It never returns a sentence or a conclusion.
"""

### M6b — The 2D / 3D viewer
**goal** one viewer, two render modes, over one precomputed pack
**in** volumes + heat · **out** `render_packs/<study>.npz`, Streamlit app

In [ ]:
# src/viz/render_pack.py

def build_render_pack(study):
    """The renderer must be DUMB: it reads one .npz and never touches DICOM, never runs a
    model, never resamples. All cost is paid offline, once.

    KEY DECISION — do not blend three anatomies into one volume.
      ANATOMY: ONE reference series (sagittal fluid-sensitive FS, present for ~91% of studies).
      HEAT:    the sagittal CAM, on that same grid. No cross-plane resampling. SHIPPED PATH.
    Why: resampling three anisotropic ANATOMIES into one grid produces a blurry pancake a
    radiologist would reject on sight. And cross-plane heat fusion needs correct DICOM affines
    for every plane — a genuine source of silent sign/axis-order bugs, for a cosmetic gain.
    Sagittal-only is one plane, one affine, no fusion, and it looks the same to the viewer.

    # STRETCH: fuse coronal + axial heat into the reference grid via the DICOM affine.
    #          Only after the sagittal path ships and only with a unit test that a known
    #          voxel maps to the same patient-space mm from all three planes.

    -> gray_iso uint8 (128,128,128), heat_iso uint8 (12,64,64,64), spacing, probs[12]
    """

# src/viz/viewer.py  — ONE viewer, a 2D/3D toggle, same pack behind both
def view_2d(pack, label):
    """Tri-planar: 3 orthogonal slices + a slider, heat overlaid. BUILD THIS FIRST —
    it is what a clinician actually reads, and it debugs the whole pipeline visually."""

def view_3d(pack, label):
    """plotly marching-cubes heat blob + MPR planes. Browser-side, survives Streamlit Cloud."""

def view_3d_demo(pack, label):   # STRETCH
    """PyVista/VTK volume rendering, PRECOMPUTED to a GIF/HTML asset.
    Interactive server-side VTK on Streamlit Cloud is CPU raycasting at seconds per frame —
    it will die on stage. Precompute the orbit, ship the file."""

---
## 5. Process for each of the 12 models

One shared encoder, twelve specialists. Each row is a different training recipe, not a
different backbone. **Silence** = % of reports that never mention the finding (measured, n=4,406).

| # | Label | Primary slot | Anatomy to look at | Prev. | Silence | Supervision | Recipe |
|---|---|---|---|---|---|---|---|
| 1 | **ACL** | `SAG_FLUID` | intercondylar notch, mid-sagittal | 20.8% | 8.1% | 🟢 clean | narrow slice window on mid-sagittal; text is clean but the pixel signal is *hard* (published AUC ~0.69) |
| 2 | **MCL** | `COR_FLUID` | medial collateral ligament | 15.3% | 9.8% | 🟢 clean | same shape as ACL; also a weak pixel signal (~0.71) |
| 3 | **Medial Meniscus** | `SAG_FLUID` | posterior horn | 40.4% | **5.7%** | 🟢 best | best-supervised label in the set — make this the first head that works end-to-end |
| 4 | **Lateral Meniscus** | `SAG_FLUID`+`COR_FLUID` | lateral posterior horn | 15.7% | 10.0% | 🟢 clean | needs laterality fix or it learns nothing |
| 5 | **Medial OA** | `COR_FLUID` | medial tibiofemoral compartment | 37.2% | 25.5% | 🟡 ok | **best pixel signal in the competition** (~0.92). Cartilage loss + osteophytes are visually obvious |
| 6 | **Lateral OA** | `COR_FLUID` | lateral compartment | 27.3% | 33.1% | 🟡 ok | mirror of #5; share weights with it, differ only in the laterality channel |
| 7 | **PF OA** | **`AX_FLUID`** | patellofemoral joint | 45.7% | 18.5% | 🟢 clean | the one head that genuinely needs axial. Only 19.4% of studies have an axial non-fluid series |
| 8 | **Effusion** | `SAG_FLUID` | suprapatellar pouch | 59.8% | 9.9% | 🟢 clean | ~0.90 from pixels. Depends entirely on M2 not per-slice-windowing |
| 9 | **Synovitis** | `SAG_FLUID` | synovial lining | 12.6% | **84.2%** | 🔴 **none** | see below — this is the interesting one |
| 10 | **Baker's** | `SAG_FLUID`+`AX_FLUID` | **popliteal fossa (posterior)** | 24.7% | 45.9% | 🟠 poor | **the 130 mm crop cuts the cyst off.** Use the `context` full-FOV profile for this head only |
| 11 | **Contusion** | `SAG_FLUID` | bone marrow oedema | 17.5% | 20.8% | 🟡 ok | STIR/FS sequences only — on non-FS series the oedema is invisible |
| 12 | **Fracture** | `SAG_FLUID` | cortical break + oedema | **7.2%** | 56.4% | 🟠 poor | rarest label. Needs `pos_weight` or the head predicts all-negative and scores 0.5 |

### The three that decide our rank

Because the metric is an unweighted mean of 12 AUCs, **a label left at chance costs the same
~0.029 as any other**. Everyone optimises Effusion and Medial OA because they respond. The
rank is decided by the three nobody can train:

- **Synovitis (84% silent).** Almost no supervision exists. Do *not* train it on the report
  labels — they are noise, and fitting noise is worse than not fitting. Options, in order:
  (a) predict from correlation with Effusion + Contusion, which co-occur with it;
  (b) semi-supervised — pseudo-label from the model's own confident predictions;
  (c) accept 0.5 and spend the time elsewhere. **Measure (a) against (c) on the gold 58 first.**
- **Fracture (7.2% prevalent, 56% silent).** Rare *and* under-reported. `pos_weight`, heavy
  augmentation, and fluid-sensitive-FS series only.
- **Baker's (46% silent).** Half the problem is ours, not the reports': our own crop removes
  the anatomy. Fixing the FOV for this one head is a cheap, real gain.

**Recommended split, revisited:** we do not need 12 *training runs*. Heads 5+6 and 3+4 are
mirror pairs — train one model with a laterality channel and read both outputs. That is
**8 distinct recipes**, not 12.

---
## 6. Staged delivery ladder

Each stage ends with something that *works*. Never be in a state where nothing runs.

| Stage | Ships | Needs | Effort | Expected score |
|---|---|---|---|---|
| **0** | a **valid submission** from metadata only — per-label prevalence as a constant, or a logistic regression on `train_series.csv` counts. No images at all. **Plus the scoring-kernel skeleton**: a notebook that walks `/kaggle/input/test_series/`, does nothing useful, and writes a correctly-shaped `submission.csv` inside the time limit. | T0 (25 MB) | **½ day** | ~0.50–0.55 |
| **1** | First **image-based** submission. `SAG_FLUID` slot only, ~1,200-study subset, frozen DINOv2-S, Ridge heads. The scoring kernel decodes real DICOM. | M2 + one CPU build + one GPU encode | 4 days | **0.68–0.76** — ~90% of the final score arrives here |
| **2** | Full 4,407 × 4 slots, mean+max pooling, rank blend, LB-budgeted A/B of laterality on/off and slot count. | Kaggle GPU ×2 | 4 days | **0.74–0.80** |
| **3** | Fine-tune one shared encoder end-to-end; rank-mean ensemble across 2 backbones. | Kaggle GPU | 4–5 days | *uncertain — only attempt if Stage 2 plateaus* |
| **Demo** | M6: 2D/3D viewer + 12 percentages + precomputed orbit assets. Runs **parallel** from day 8; does not wait for Stage 2. | laptop | 5 days | graded, not scored |

**Calibrate expectations.** LB top ≈ **0.952**, 15th ≈ 0.943 — not reachable in a few weeks. A
fully-developed *public* pipeline reports macro **0.760**. A realistic target for us is
**0.74–0.80**. Gate each stage on *relative* movement — Stage 2 must beat Stage 1 by more than
the fold-to-fold spread — never on hitting an absolute number.

**Spend the 5 daily submissions deliberately**, because the LB is the only honest arbiter:
Mon Stage-0 baseline · Wed Stage-1 image arm · Thu laterality ON vs OFF (same seed, same folds)
· Fri 3-slot vs 4-slot. Log every one in `lb_log.csv` with its OOF in a **separate column** —
OOF measures agreement with an LLM, LB measures agreement with radiologists. Never quote them
on the same ladder.

**Do Stage 0 on day one.** A valid submission in the leaderboard removes all
end-of-project submission-format panic, and it is genuinely half a day of work.

**And time the scoring kernel from day one.** Every stage after this must answer "does it still
fit in 9 hours for 1,300 studies?". Measure `per_study_seconds` on 20 studies and multiply —
discovering at Stage 3 that inference takes 14 hours means throwing away Stage 3.

---
## 7. Who builds what

Contracts from §2 are the handoff points, so nobody blocks anybody.

| Person | Modules | First deliverable | Hands over |
|---|---|---|---|
| **A** | M1 + M2 | `manifest_series.parquet` + 58 gold volumes | `.npy` volumes + manifest |
| **B** | M3 | `soft_targets.csv` + `label_audit.csv` | `y`, `w`, `folds` |
| **C** | M4 + M5 | cached features, then heads | OOF preds + `submission.csv` |
| **D** | M6a + M6b | 2D tri-planar viewer on gold 58 | the app |

**Unblocking trick:** D should not wait for C. Generate a **fake heat volume** (a Gaussian
blob at a plausible location) that satisfies the M6a contract, and build the entire viewer
against it. When the real CAMs arrive it is a one-line swap. Same for B → C: ship
`soft_targets.csv` full of random values on day 1 so C can build the training loop immediately.

---
## 8. Dependencies to add

Grouped by stage — do not install stage 3 on day 1.

```txt
# Stage 1 — preprocessing (laptop, Apple Silicon, py3.10.6)
pydicom>=3.0,<4
pylibjpeg>=2.0                # MANDATORY: JPEG-Lossless + JPEG-2000 series exist in this corpus
pylibjpeg-libjpeg>=2.1        # bare pydicom raises on .pixel_array without these
pylibjpeg-openjpeg>=2.4
pyarrow>=16,<21               # parquet manifests
opencv-python-headless        # cv2.resize INTER_AREA — ~10x faster than scipy.ndimage.zoom
python-dotenv                 # already used by kaggleFetch.ipynb
kaggle

# Stage 2 — models
torch>=2.5,<3                 # <2.5 pins numpy<2 and breaks our numpy 2.2.6. MPS works on M-series
timm>=1.0
transformers>=4.44            # AutoModel for dinov2, with local_files_only=True on Kaggle

# Stage 3 / demo — visualisation
streamlit>=1.40
plotly>=6.0                   # browser-side 3D, survives Streamlit Cloud
pyvista>=0.48                 # STRETCH: precomputed orbit assets only, NOT interactive server-side
```

---
## 9. Open questions — each with a default so nothing blocks

1. **Do we pull the 17.4 GB preprocessed mirror, or preprocess DICOM ourselves?**
   → *Default: both.* Mirror for Stage 2 speed; our own M2 on the gold 58 so we control
   the pipeline and can build the `context` profile the mirror does not have.
2. **Frozen features or fine-tune?** → *Default: frozen for Stage 2, fine-tune for Stage 3.*
   Frozen ImageNet/DINOv2 features are **not** MSK-MRI features and will underperform
   fine-tuning by a wide margin. Frozen is scaffolding, not the destination.
3. **How many slices K per plane?** → *Default: 24.* Median series is 30 slices; 24 keeps the
   feature cache at ~600 MB and fits Kaggle's 20 GB `/kaggle/working`.
4. **Synovitis: attempt it or accept 0.5?** → *Default: measure the Effusion-correlation
   approach on the gold 58 (one afternoon), and only then decide.*
5. **Does anyone on the team have a CUDA machine?** → *Default: assume no.* All training is
   planned for Kaggle GPU with a 9-hour ceiling.
6. **Streamlit Cloud or local demo?** → *Default: local for demo day, Cloud as a bonus.*
   Precomputed assets mean the demo cannot fail live.

**One 15-minute task this week** that changes the dependency list: in an internet-off Kaggle
kernel, check whether pydicom 3.x's native decoders already handle this corpus —
`from pydicom.pixels.decoders import JPEG2000LosslessDecoder as D; print(D.is_available)`.
If they do, `pylibjpeg-*` drops out of the requirements entirely and the offline-install
problem disappears. If they don't, we must pre-attach the wheels as a Kaggle Dataset, because
the scoring kernel has no internet and cannot `pip install`.

---

### Reminders that will save a week

- The scoring kernel has **no internet**. Every weight must be a pre-attached Kaggle Model.
- **9-hour** GPU limit, hidden test ≈1,300 studies → budget `per_study_seconds × 1300`.
- `test.csv` has **no Report column**. Reports are a training-time teacher only.
- The metric reads **rank order only** — ensemble by rank, never by probability.
- Fix **laterality** before training, or 4 of 12 heads learn from an axis they cannot see.

---
## Appendix — original brief (kept verbatim)

> Context: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data
> data set of 820 000 images of MRI of knees from kaggle dataset
> we need to create a pre-processing pipeline (using pre-trained model, such as DINOv2 /
> MRNet/ResNet Backbones / ...) that we can attach to our own custom pipeline.
>
> the database has:
> **train.csv** One row per training study — StudyInstanceUID, Report (free-text, any of
> several languages), and twelve binary labels: ACL, MCL, Medial Meniscus, Lateral Meniscus,
> Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.
>
> **train_series.csv** One row per training series — StudyInstanceUID, SeriesInstanceUID,
> Fluid_Sensitive, Fat_Suppression, Anatomical_Plane (Sagittal, Coronal, or Axial).
>
> - we might to have for each label a specified model running in order to get the best score possible.
> - expected output: for each serie: we have 3 array of images (one per axis: X = 40 ~ images /
>   Y = 40 ~ images and Z = 40 ~ images). the preprocessing should categoryze / classify the images.
>   the specific models should be able to return a knee with color spot on injure.
> - also we should be able to compile images set (X/Y and Z) — the whole sequence of a serie —
>   into 3D visualisation

*(§0 lists where this brief needed correcting; §5 is the answer to "a specified model per label";
§6b is the answer to the 3D visualisation.)*